# Data Cleaning Walkthrough

**Note:** this notebook runs on a synthetic sample dataset that reproduces the same problems
found in the real files: inconsistent customer names, mixed date formats, currency symbols
stored as text, duplicate line items and out-of-range values.

Three stages are shown below:

1. Profiling the raw data
2. Finding duplicate customers with fuzzy matching
3. Cleaning, validating and building the exceptions report

## 1. Load the raw data and see what we are dealing with

In [1]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(7)

# ---- build a messy sample export (stands in for the client's Excel files) ----
customers = ["Sharma Traders", "sharma traders", "SHARMA TRADERS PVT LTD", "Sharma Tradrs",
             "Verma & Sons", "verma and sons", "Verma Sons Ltd", "Nova Supplies",
             "NOVA SUPPLIES", "Nova Suplies Co", "Kiran Enterprises", "kiran enterprise",
             "Deep Metals", "Deep Metal Works"]
regions   = ["North", "north", "N", " South", "South", "S", "East", "West", "west", ""]
products  = ["Steel Pipe 2in", "steel pipe 2in ", "Copper Wire 5m", "Copper Wire 5m",
             "Valve Assembly", "valve assembly", "Gasket Set", "Gasket Set "]
statuses  = ["Completed", "completed", "COMPLETED", "Cancelled", "cancelled", "Pending", "Returned"]
dates     = ["14/03/2026", "2026-03-14", "05/04/2026", "2026-04-05", "18-Mar-26", "45001", ""]

n = 420
raw = pd.DataFrame({
    "Order ID":       [f"{100000 + int(rng.integers(0, 260)):06d}" for _ in range(n)],
    " Customer Name": rng.choice(customers, n),
    "Region":         rng.choice(regions, n),
    "Product":        rng.choice(products, n),
    "Order Date":     rng.choice(dates, n, p=[.34, .22, .16, .12, .09, .04, .03]),
    "Qty":            rng.choice(["1", "2", "3", "5", "10", "-2", ""], n,
                                 p=[.28, .24, .18, .14, .10, .03, .03]),
    "Unit Price":     [f"Rs {rng.integers(50, 4000):,}" for _ in range(n)],
    "Status":         rng.choice(statuses, n),
})
raw.loc[raw.sample(6, random_state=1).index, "Unit Price"] = "approx 1200"
raw.loc[len(raw)] = ["", "Grand Total", "", "", "", "", "", ""]

raw.to_csv("sales_raw_sample.csv", index=False)

# ---- load it the safe way: everything as text, nothing auto-converted ----
df = pd.read_csv("sales_raw_sample.csv", dtype=str, keep_default_na=False)

print(f"rows loaded: {len(df):,}   columns: {len(df.columns)}")
df.head(12)

rows loaded: 421   columns: 8


,Order ID,Customer Name,Region,Product,Order Date,Qty,Unit Price,Status
0,100245,NOVA SUPPLIES,North,Valve Assembly,14/03/2026,5,"Rs 1,038",cancelled
1,100162,Verma & Sons,S,Valve Assembly,2026-03-14,2,"Rs 3,138",cancelled
2,100177,kiran enterprise,S,steel pipe 2in,05/04/2026,1,"Rs 1,669",Pending
3,100233,Verma & Sons,West,Gasket Set,14/03/2026,5,"Rs 3,050",COMPLETED
4,100150,SHARMA TRADERS PVT LTD,north,Valve Assembly,2026-03-14,,approx 1200,Returned
5,100201,sharma traders,North,Steel Pipe 2in,05/04/2026,1,"Rs 2,717",Cancelled
6,100216,Verma & Sons,West,steel pipe 2in,05/04/2026,5,Rs 527,Cancelled
7,100058,NOVA SUPPLIES,West,Copper Wire 5m,14/03/2026,1,Rs 119,Returned
8,100014,sharma traders,West,valve assembly,05/04/2026,2,"Rs 2,814",Returned
9,100078,kiran enterprise,North,Gasket Set,14/03/2026,3,Rs 956,completed


### Profiling: frequency counts find most of the problems

Before changing anything, count the values in every column that is supposed to be categorical.
This one step exposes casing variants, stray whitespace, single-letter codes and blanks.

In [2]:
df.columns = (df.columns.str.strip().str.lower()
                .str.replace(r"[^\w\s]", "", regex=True)
                .str.replace(r"\s+", "_", regex=True))
df = df.rename(columns={"qty": "quantity"})

for col in ["region", "status"]:
    print(f"--- {col} ---")
    print(df[col].value_counts().to_string())
    print()

profile = pd.DataFrame({
    "blank_or_empty": df.apply(lambda s: (s.str.strip() == "").sum()),
    "distinct_values": df.nunique(),
    "sample_values": pd.Series({c: df.loc[df[c].str.strip() != "", c].head(3).tolist()
                                for c in df.columns}),
})
profile

--- region ---
region
 South    49
West      48
South     45
North     43
north     43
N         43
west      42
S         37
          36
East      35

--- status ---
status
completed    73
Pending      68
cancelled    63
COMPLETED    59
Returned     53
Cancelled    53
Completed    51
              1



,blank_or_empty,distinct_values,sample_values
order_id,1,213,"[100245, 100162, 100177]"
customer_name,0,15,"[NOVA SUPPLIES, Verma & Sons, kiran enterprise]"
region,36,10,"[North, S, S]"
product,1,8,"[Valve Assembly, Valve Assembly, steel pipe 2in ]"
order_date,15,7,"[14/03/2026, 2026-03-14, 05/04/2026]"
quantity,13,7,"[5, 2, 1]"
unit_price,1,403,"[Rs 1,038, Rs 3,138, Rs 1,669]"
status,1,8,"[cancelled, cancelled, Pending]"


## 2. Finding duplicate customers with fuzzy matching

The same business appears under several spellings, which inflates the customer count.
Exact matching cannot catch this. `rapidfuzz` generates candidate pairs above a similarity
threshold, and a human confirms them. The algorithm suggests, it never decides.

In [3]:
from rapidfuzz import process, fuzz

df["customer_key"] = (df["customer_name"].str.strip().str.lower()
                        .str.replace(r"[^\w\s]", "", regex=True)
                        .str.replace(r"\b(pvt|ltd|limited|co|and)\b", "", regex=True)
                        .str.replace(r"\s+", " ", regex=True).str.strip())

names = sorted(k for k in df["customer_key"].unique() if k and k != "grand total")
print(f"distinct customer spellings before matching: {len(names)}")

pairs = set()
for name in names:
    for match, score, _ in process.extract(name, names, scorer=fuzz.token_sort_ratio,
                                           limit=5, score_cutoff=80):
        if match != name:
            pairs.add((*sorted([name, match]), round(score, 1)))

review = (pd.DataFrame(sorted(pairs), columns=["name_a", "name_b", "similarity"])
            .sort_values("similarity", ascending=False)
            .reset_index(drop=True))
review["same_customer_yn"] = ""       # client fills this column in
review

distinct customer spellings before matching: 9


,name_a,name_b,similarity,same_customer_yn
0,kiran enterprise,kiran enterprises,97.0,
1,sharma traders,sharma tradrs,96.3,
2,nova suplies,nova supplies,96.0,
3,deep metal works,deep metals,81.5,


## 3. Clean, validate, and report the exceptions

Dates are parsed with explicit formats in passes rather than by inference, because inference can
read `05/04/2026` as April in one row and May in another. Bad rows are flagged and exported,
not silently dropped.

In [4]:
rows_in = len(df)

# drop total rows and rows with no order id
df = df[~df["customer_name"].str.strip().str.lower().isin(["grand total", "total"])]
df = df[df["order_id"].str.strip() != ""]

# dates: explicit formats, in passes
d1 = pd.to_datetime(df["order_date"], format="%d/%m/%Y", errors="coerce")
d2 = pd.to_datetime(df["order_date"], format="%Y-%m-%d", errors="coerce")
d3 = pd.to_datetime(df["order_date"], format="%d-%b-%y", errors="coerce")
df["order_date_clean"] = d1.fillna(d2).fillna(d3)

# excel serial numbers that lost their formatting
serial = df["order_date"].str.match(r"^\d{5}$") & df["order_date_clean"].isna()
df.loc[serial, "order_date_clean"] = pd.to_datetime(
    df.loc[serial, "order_date"].astype(int), unit="D", origin="1899-12-30")

# numbers: strip symbols, coerce, keep track of what failed
def clean_numeric(s):
    return pd.to_numeric(s.str.replace(r"[^\d.\-]", "", regex=True).replace("", None),
                         errors="coerce")

df["quantity"]   = clean_numeric(df["quantity"])
df["unit_price"] = clean_numeric(df["unit_price"])

# text standardisation
df["region"] = (df["region"].str.strip().str.title()
                  .replace({"N": "North", "S": "South", "E": "East", "W": "West", "": "Unknown"}))
df["status"] = df["status"].str.strip().str.title()
product_map = {"steel pipe 2in": "Steel Pipe 2in", "copper wire 5m": "Copper Wire 5m",
               "valve assembly": "Valve Assembly", "gasket set": "Gasket Set"}
df["product"] = df["product"].str.strip().str.lower().map(product_map)

# duplicates: the real key is order + product, not order alone
dupes_found = df.duplicated(subset=["order_id", "product"]).sum()
df = df.drop_duplicates(subset=["order_id", "product"], keep="last")

# ---- exceptions report ----
exceptions = []
def flag(mask, reason):
    if mask.any():
        out = df.loc[mask].copy()
        out["exception_reason"] = reason
        exceptions.append(out)

flag(df["order_date_clean"].isna(), "date could not be parsed")
flag(df["quantity"].isna(), "quantity missing or not numeric")
flag(df["quantity"] < 0, "negative quantity")
flag(df["unit_price"].isna(), "unit price could not be parsed")

exceptions_df = pd.concat(exceptions, ignore_index=True)

summary = pd.DataFrame({
    "stage": ["rows loaded", "total rows removed", "duplicate line items removed",
              "rows flagged for review", "clean rows remaining"],
    "count": [rows_in, rows_in - len(df), dupes_found, len(exceptions_df), len(df)],
})

print(exceptions_df["exception_reason"].value_counts().to_string())
print()
summary

exception_reason
date could not be parsed           13
quantity missing or not numeric    10
negative quantity                   9



,stage,count
0,rows loaded,421
1,total rows removed,72
2,duplicate line items removed,71
3,rows flagged for review,32
4,clean rows remaining,349


In [5]:
clean = df.loc[
    df["order_date_clean"].notna() & df["quantity"].notna() & df["unit_price"].notna(),
    ["order_id", "order_date_clean", "customer_key", "region", "product", "quantity",
     "unit_price", "status"]
].rename(columns={"order_date_clean": "order_date", "customer_key": "customer_name"})

clean["line_total"] = clean["quantity"] * clean["unit_price"]

assert clean["order_date"].notna().all(), "unparsed dates in output"
assert not clean.duplicated(subset=["order_id", "product"]).any(), "duplicate line items"
print("validation passed")

clean.head(10)

validation passed


,order_id,order_date,customer_name,region,product,quantity,unit_price,status,line_total
1,100162,2026-03-14,verma sons,South,Valve Assembly,2.0,3138,Cancelled,6276.0
5,100201,2026-04-05,sharma traders,North,Steel Pipe 2in,1.0,2717,Cancelled,2717.0
6,100216,2026-04-05,verma sons,West,Steel Pipe 2in,5.0,527,Cancelled,2635.0
10,100074,2023-03-16,verma sons,East,Gasket Set,3.0,314,Completed,942.0
13,100001,2026-03-14,deep metals,South,Copper Wire 5m,2.0,2826,Cancelled,5652.0
15,100213,2026-03-14,kiran enterprise,South,Valve Assembly,2.0,1576,Completed,3152.0
16,100034,2026-04-05,verma sons,North,Copper Wire 5m,2.0,2807,Completed,5614.0
18,100030,2023-03-16,kiran enterprise,South,Valve Assembly,2.0,2256,Completed,4512.0
21,100078,2026-04-05,deep metals,West,Copper Wire 5m,1.0,143,Completed,143.0
22,100088,2026-04-05,verma sons,Unknown,Valve Assembly,2.0,3566,Completed,7132.0
